## <font color="#FF9900"> Fluxo do Kafka local (Docker)</font>
<font size="3" color="gray">optei por utilizar o Docker para rodar o Kafka no meu próprio computador antes de subir para o AWS
utilizarei o docker-compose.yml e o serviço de WSL para criar um servidor local e rodar o kafka na minha própria máquina.
Vou deixar aqui um tutorial de instalação do WSL e o link para o Docker desktop caso alguem queira tentar usar



</font>



[AULA WSL](https://www.youtube.com/watch?v=trto4i0Olwg&t=2s)  <BR>
[DOCKER DESKTOP](https://www.docker.com/products/docker-desktop/)


In [ ]:
import os
import time
import boto3
import dotenv


dotenv.load_dotenv()

In [ ]:
import time
from confluent_kafka.admin import AdminClient, NewTopic

admin_client = AdminClient({'bootstrap.servers': 'localhost:9092'})

# Apaga o tópico com todas as mensagens acumuladas
fs = admin_client.delete_topics(['transactions'], operation_timeout=15)
for topic, future in fs.items():
    try:
        future.result()
        print(f"'{topic}' deletado!")
    except Exception as e:
        print(f"{e}")

# Aguarda o Kafka processar a deleção internamente

time.sleep(3)

#  Recria o tópico limpo com 3 partições
print("Recriando tópico 'transactions'")
fs = admin_client.create_topics([NewTopic("transactions", num_partitions=3, replication_factor=1)])
for topic, future in fs.items():
    try:
        future.result()
        print(f"Tópico '{topic}' recriado")
    except Exception as e:
        print(f"{e}")


In [ ]:
#Instalar o Confluent kafka para utilização da ferramenta e fazer os imports necessarios 
%pip install confluent-kafka
%pip install  pyspark

import json
import time
import random
from datetime import datetime
import pandas as pd

from pathlib import Path
from confluent_kafka import Producer,Consumer, KafkaError


from src.data.utils import enriquecer_alunos_silver, carregar_parquet_local, preparar_dimensoes_silver

In [ ]:
# Configurações do Kafka Local
TOPIC_NAME = "transactions"
PATH_BRONZE = Path("../data/bronze")
PATH_PRATA = Path("../data/prata")
ANO = 2023
KAFKA_SERVER = "localhost:9092"


In [ ]:
 
def delivery_report(err, msg):
    """
    Retorna um log de envio ou erro do producer Kafka 
    """

    if err is not None:
        print(f"[ERRO] Falha no envio: {err}")
    else:
        print(f"[SUCESSO] Aluno enviado para {msg.topic()} | Partição: {msg.partition()} | ID Aluno: {msg.key().decode('utf-8')}")


<font  size="3" color="gray">

Irei configurar o Producer 
que é responsavel por enviar a mensagem para o Kafka <br>
Para isso vamos simular um serviço em streaming, em que a cada segundo um aluno entrega a prova, 
<br> criando uma linha no TS_ALUNO 

O Objetivo dessa parte é carregar o dataframe de alunos e enviar para o Kafka 
O Script faz o papel de producer nesse caso 
ao chegar o kafka o consumer recuperará as linhas geradas pelo producer 
irá ler cada uma delas e fazer o pivot table com o municipio e o aluno, gerando assim a camada silver 

</font>

In [ ]:
# Defina aqui o ano que deseja processar (2023, 2024 ou 2025)
ANOS_PROVAS = [2023]#,2024,2025 ]

KAFKA_SERVER = "localhost:9092"
TOPIC_NAME = "transactions"
PATH_BRONZE = Path("../data/bronze")

# Carrega o arquivo inteiro de alunos do ano escolhido

for ANO in ANOS_PROVAS:
    caminho_parquet = PATH_BRONZE / f"ano={ANO}/dados/TS_ALUNO.parquet"
    print(f"[INFO] Carregando base de alunos do ano {ANO} de: {caminho_parquet.name}...")
    df_alunos_source = pd.read_parquet(caminho_parquet)


    df_alunos_source = df_alunos_source.head(1000)


    total_registros = len(df_alunos_source)
    print(f"Total de registros para enviar ao Kafka: {total_registros}")

    producer = Producer({'bootstrap.servers': KAFKA_SERVER})

    print(f"\n Iniciando streaming do ano {ANO} (Disparando em velocidade máxima)...")

    try:
        for index, row in df_alunos_source.iterrows():
            dados_aluno = row.to_dict()
            dados_aluno_limpo = {}
            for k, v in dados_aluno.items():
                if pd.isna(v):
                    dados_aluno_limpo[k] = None
                elif hasattr(v, 'item'):
                    dados_aluno_limpo[k] = v.item()
                else:
                    dados_aluno_limpo[k] = v

            id_aluno = str(dados_aluno_limpo.get('ID_ALUNO') or index)
            
            producer.produce(
                TOPIC_NAME,
                key=id_aluno.encode('utf-8'),
                value=json.dumps(dados_aluno_limpo).encode('utf-8'),
                callback=delivery_report
            )
            
            # Faz o poll a cada 500 mensagens para liberar memória interna do cliente
            if index % 500 == 0:
                producer.poll(0)
                print(f"📡 Status: {index}/{total_registros} mensagens enviadas...")

    except KeyboardInterrupt:
        print("\n Envio pausado pelo usuário.")
    finally:
        print(" Finalizando envios pendentes (flush)...")
        producer.flush()
        print(f" Ingestão concluída com sucesso para o ano {ANO}!")


In [ ]:
def processar_camada_silver(ano: int | str, df_alunos: pd.DataFrame = None, path_bronze: Path = Path("../data/bronze")) -> pd.DataFrame:
    """
    Processa e integra a camada Silver. 
    Se df_alunos for informado, processa-o (Streaming).
    Caso contrário, carrega o arquivo Parquet do disco local (Batch).
    """
    # Se for Batch (df_alunos é None), carrega do disco local
    if df_alunos is None:
        print(f"\n[INFO] Modo Batch: Carregando alunos do ano {ano} do disco...")
        df_alunos = pd.read_parquet(path_bronze / f"ano={ano}/dados/TS_ALUNO.parquet")
        
    # Prepara as tabelas de referência
    municipio_dim, uf_dim = preparar_dimensoes_silver(ano, path_bronze)
    
    # Executa o enriquecimento comum (Junções e Cálculos)
    df_silver = enriquecer_alunos_silver(df_alunos, municipio_dim, uf_dim)
    
    return df_silver


<font size='3' color='gray'>O Consumer opera através das mensagens recebidas pelo Kafka 
</font>


In [ ]:
KAFKA_SERVER = "localhost:9092"
TOPIC_NAME = "transactions"
PATH_BRONZE = Path("../data/bronze")
PATH_PRATA = Path("../data/prata")
ANO = 2023

# Configurações do Micro-batch
BATCH_SIZE_LIMIT = 10000 #LIMITE DE MENSAGENS POR BUFFER
BATCH_TIME_LIMIT = 10 # TEMPO LIMITE PARA FECHAR O BUFFER  

# Carrega e prepara as dimensões usando o seu utils.py
print("[INFO] Preparando tabelas de referência (Município/UF)...")
municipio_dim, uf_dim = preparar_dimensoes_silver(ANO, PATH_BRONZE)

# Configura e assina o Consumidor Kafka
consumer = Consumer({
    'bootstrap.servers': KAFKA_SERVER,
    'group.id': 'pipeline-silver-group-v3',
    'auto.offset.reset': 'earliest'
})
consumer.subscribe([TOPIC_NAME])
print("Consumidor iniciado e escutando o Kafka...")

buffer_mensagens = []
ultimo_envio_tempo = time.time()

# Loop de processamento
try:
    while True:
        msg = consumer.poll(0.5)
        
        if msg is not None:
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"[Erro] Kafka: {msg.error()}")
                    break
            else:
                dados_aluno_json = json.loads(msg.value().decode('utf-8'))
                
                colunas_obrigatorias = ['NU_ANO_AVALIACAO', 'CO_UF', 'CO_MUNICIPIO', 'TP_SERIE']
                if all(col in dados_aluno_json for col in colunas_obrigatorias):
                    buffer_mensagens.append(dados_aluno_json)
        
        tempo_decorrido = time.time() - ultimo_envio_tempo
        tamanho_lote = len(buffer_mensagens)
        
        if tamanho_lote > 0 and (tamanho_lote >= BATCH_SIZE_LIMIT or tempo_decorrido >= BATCH_TIME_LIMIT):
            df_lote_alunos = pd.DataFrame(buffer_mensagens)
            df_lote_alunos = df_lote_alunos.dropna(subset=colunas_obrigatorias)
            
            if not df_lote_alunos.empty:
                for col_name in colunas_obrigatorias:
                    df_lote_alunos[col_name] = pd.to_numeric(df_lote_alunos[col_name]).astype(int)
                if 'VL_PROFICIENCIA_LP' in df_lote_alunos.columns:
                    df_lote_alunos['VL_PROFICIENCIA_LP'] = pd.to_numeric(df_lote_alunos['VL_PROFICIENCIA_LP'])
                
 
                # Transformar as tabelas para o padrão Silver 
                df_silver = enriquecer_alunos_silver(df_lote_alunos, municipio_dim, uf_dim)
                
                # Grava o lote consolidado em um único arquivo Parquet
                pasta_destino = PATH_PRATA / f"ano={ANO}"
                pasta_destino.mkdir(parents=True, exist_ok=True)
                
                timestamp_atual = int(time.time())
                caminho_arquivo = pasta_destino / f"lote_alunos_{timestamp_atual}.parquet"
                
                df_silver.to_parquet(caminho_arquivo, index=False)
               # print(f"LOTE GRAVADO na camada Silver: {caminho_arquivo.name} (Tamanho: {len(df_silver)} registros)")
            else:
                print("O lote continha apenas registros inválidos (com chaves nulas) e foi ignorado.")
            #
            buffer_mensagens = []
            ultimo_envio_tempo = time.time()

except KeyboardInterrupt:
    print("\n Consumidor parado pelo usuário.")
finally:
    consumer.close()


<font size="3" color="gray">Rodar no terminal para ler o status das mensagens processadas 
</font> 
<br>
<font size="3" color="green">
docker exec -it kafka-fiap /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group pipeline-silver-group-multiyear
</font>

In [ ]:
!docker exec -it kafka-fiap /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group pipeline-silver-group-v3
